# Laboratorium 4: od prostej do interakcji

Jak przewidywać sprzedaż na podstawie budżetów reklamowych? Zaczniemy od prostej,
dodamy kolejne cechy, sprawdzimy hipotezy o ich współczynnikach i zbudujemy model z interakcją.

[Karta laboratorium](https://rodakt.github.io/MAD/laby/04-regresja-liniowa.html)

Zapisz kopię notebooka na swoim Dysku (*Plik → Zapisz kopię na Dysku*).
Kod wpisuj w pustych komórkach. Podpowiedzi otwieraj wtedy, gdy ich potrzebujesz.
Przed obliczeniem spróbuj przewidzieć wynik, a po obliczeniu zapisz krótką interpretację
w komórce „Twoja odpowiedź” (dwuklik pozwala ją edytować).

Ćwiczenia oznaczone jako opcjonalne są dla osób, które skończą wcześniej.

## Przygotowanie

Uruchom poniższą komórkę. Wczytuje dane *Advertising* z rozdziału 6 książki
i dzieli je na 150 rynków uczących (`data_tr`) oraz 50 testowych (`data_te`).
Na razie pracujemy tylko na `data_tr`; `data_te` otworzymy w części 5.
Ten sam podział zachowujemy przez całe zajęcia.

Każdy wiersz opisuje jeden rynek. Budżety reklamy `TV`, `radio`, `newspaper` są podane
w tysiącach USD, a sprzedaż `sales` w tysiącach sztuk.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split

advertising = pd.read_csv("https://rodakt.github.io/MAD/laby/dane/Advertising.csv", index_col=0)
data_tr, data_te = train_test_split(advertising, test_size=0.25, random_state=2026)

## 1. Od stałej do prostej

### 1.1. Dane i pytanie

Wyświetl pięć pierwszych rynków uczących i odczytaj jeden wiersz w dolarach i sztukach.
Narysuj sprzedaż względem budżetu TV, opisując osie wraz z jednostkami.
Czy prosta wydaje się rozsądnym przybliżeniem? Czy rynki o podobnym budżecie zawsze mają podobną sprzedaż?

<details>
<summary>Podpowiedź 1: tabela</summary>

`data_tr.head()` pokazuje pierwsze wiersze, a `data_tr["TV"]` wybiera kolumnę.

</details>

<details>
<summary>Podpowiedź 2: wykres</summary>

`plt.scatter(x, Y)` rysuje punkty. Osie opisują `plt.xlabel("opis osi")` i `plt.ylabel("opis osi")`. Na końcu użyj `plt.show()`.

</details>

**Twoja odpowiedź:**

### 1.2. Stała predykcja jako punkt odniesienia

Na początek przyjmij umownie, że na każdym rynku przewidujesz sprzedaż 10 tys. sztuk.
Za chwilę sprawdzisz, czy można wybrać lepszą wspólną predykcję. Oblicz jedną resztę
(obserwacja minus predykcja), a potem błędy `errors` na wszystkich rynkach uczących.
Oblicz średnią kwadratów błędów, czyli MSE.

Następnie oblicz średnią sprzedaż uczącą `mean_tr` i MSE przewidywania tej wartości, `mse_mean_tr`.
Dlaczego zwykła średnia błędów ze znakami nie wystarczyłaby do oceny wielkości pomyłek?
Dlaczego warto zachować wynik modelu stałego do porównania z bardziej rozbudowanymi modelami?

<details>
<summary>Podpowiedź 1: operacje na kolumnie</summary>

Od kolumny można odjąć liczbę. `errors**2` podnosi każdy element do kwadratu, a `.mean()` oblicza średnią.

</details>

<details>
<summary>Podpowiedź 2: jedna obserwacja</summary>

`data_tr["sales"].iloc[0]` wybiera pierwszą sprzedaż w tabeli.

</details>

**Twoja odpowiedź:**

### 1.3. Własna prosta

Przyjmij model $\widehat y = a + bx$, gdzie $x$ to budżet TV.
Przyjmij `a = 5`, `b = 0.04`. Przed uruchomieniem oblicz w myślach predykcję dla $x=100$
i zapisz swój wynik. Następnie wpisz parametry i oblicz w Pythonie `a + b * 100`.
Czy wynik zgadza się z Twoim obliczeniem? Podaj przewidywaną sprzedaż w sztukach.

Oblicz teraz predykcje `Y_hat` na próbie uczącej, ich MSE i dodaj prostą do wykresu punktowego.

Przewidź, co zmieni zwiększenie tylko `a`, a co zwiększenie tylko `b`. Sprawdź na jednej lub dwóch zmianach.
Zachowaj wybrane parametry jako `a_manual`, `b_manual` i ich MSE jako `mse_manual`.
Czy udało się pobić przewidywanie średniej? Nie musisz znaleźć minimum.

<details>
<summary>Podpowiedź 1: predykcje</summary>

W równaniu zastąp pojedynczy budżet całą kolumną `data_tr["TV"]`.

</details>

<details>
<summary>Podpowiedź 2: rysowanie prostej</summary>

`grid = np.array([0, 300])` tworzy dwa budżety. `plt.plot(grid, a + b * grid)` łączy odpowiadające im punkty. Dodaj tę linię przed `plt.show()`.

</details>

**Twoja odpowiedź:**

### 1.4. Dopasowanie OLS

Czy procedura minimalizująca MSE na tych samych danych może znaleźć prostą lepszą od Twojej?
Dopasuj `sales ~ TV` na próbie uczącej jako `model_tv`. Odczytaj współczynniki,
oblicz predykcje i ich MSE `mse_tv_tr`. Porównaj ze swoją prostą i modelem stałym.

Oblicz ze współczynników predykcję dla TV = 100 tys. USD.
Zinterpretuj nachylenie: jaka różnica sprzedaży, w sztukach, odpowiada budżetowi większemu o 1000 USD?
Czy wynik dowodzi skutku zwiększenia wydatków?

Biblioteka nazywa wyraz wolny `Intercept`, a nachylenie `TV`.
Odpowiadają one Twoim `a`, `b` oraz $\widehat\beta_0$, $\widehat\beta_1$ z książki.

<details>
<summary>Podpowiedź 1: dopasowanie</summary>

`smf.ols("sales ~ TV", data=data_tr).fit()` dopasowuje model z wyrazem wolnym. Po lewej stronie `~` jest wynik, po prawej cecha.

</details>

<details>
<summary>Podpowiedź 2: współczynniki i predykcje</summary>

`model_tv.params` zawiera współczynniki, np. `model_tv.params["TV"]`. `model_tv.predict(data_tr)` zwraca predykcje.

</details>

**Twoja odpowiedź:**

## 2. Regresja wieloraka: radio ponad TV

### 2.1. Dodaj drugą cechę

Co może wnieść znajomość budżetu radia, jeśli znamy już budżet TV?
Zaproponuj formułę z obiema cechami i dopasuj model jako `model_add`.
Odczytaj współczynniki, oblicz uczące MSE `mse_add_tr` i porównaj z modelem TV.

Czy po dodaniu cechy minimalna suma kwadratów reszt na tych samych danych może wzrosnąć?
Wyjaśnij, dlaczego to porównanie nie rozstrzyga jeszcze o jakości predykcji na nowych rynkach.

<details>
<summary>Podpowiedź: kilka cech</summary>

W formule kolejne cechy łączymy znakiem `+`. Wyraz wolny jest dodawany domyślnie.

</details>

**Twoja odpowiedź:**

### 2.2. Cztery rynki: co oznacza współczynnik przy TV?

Utwórz tabelę `markets` jako `pd.DataFrame`, zawierającą cztery rynki w podanej kolejności:

| Wiersz | TV | radio |
|---|---:|---:|
| 0 | 100 | 10 |
| 1 | 110 | 10 |
| 2 | 100 | 40 |
| 3 | 110 | 40 |

W każdej parze budżet TV rośnie ze 100 do 110 tys. USD, a budżet radia pozostaje stały.
Przed obliczeniem przewidź: czy różnica sprzedaży będzie taka sama przy radiu = 10 i radiu = 40?

1. Oblicz predykcje `pred_add` modelu `model_add` dla czterech rynków.
2. Oblicz różnice predykcji 1−0 i 3−2. Porównaj je z dziesięciokrotnością współczynnika TV;
   wszystkie wyniki podaj w sztukach.
3. Na tej podstawie zinterpretuj współczynnik TV przy ustalonym budżecie radia, a następnie
   współczynnik radia przy ustalonym budżecie TV.

Model, w którym różnica predykcji związana ze zmianą jednej cechy nie zależy od wartości drugiej,
nazywamy **addytywnym**. Równość porównywanych przyrostów wynika z postaci tego modelu.

Na koniec porównaj współczynnik TV w `model_tv` i `model_add`.
W którym z tych modeli porównanie zakłada ustalony budżet radia?

<details>
<summary>Podpowiedź 1: nowa tabela</summary>

`pd.DataFrame({"TV": [100, 110, 100, 110], "radio": [10, 10, 40, 40]})` tworzy tabelę czterech rynków.

</details>

<details>
<summary>Podpowiedź 2: konkretna predykcja</summary>

`pred_add.iloc[0]` wybiera pierwszą predykcję, `.iloc[1]` drugą.

</details>

**Twoja odpowiedź:**

## 3. Prasa, test $t$ i test $F$

Czy niższy błąd uczący wystarcza, żeby uznać dodatkową cechę za potrzebną?
Sprawdzimy konkretne hipotezy o współczynnikach.

Klasyczne testy z tej części opierają się na założeniach modelu liniowego z rozdziału 6:
poprawnej postaci średniej, nieskorelowanych błędach i ich stałej wariancji.
Dokładne rozkłady $t$ i $F$ wymagają ponadto normalności błędów.
Wnioski formułujemy w ramach przyjętego modelu; samo dopasowanie nie sprawdza tych założeń.

### 3.1. Czy prasa wnosi coś ponad TV i radio?

Dopasuj `model_full` z trzema cechami: `TV`, `radio`, `newspaper`.
Zapisz hipotezę $H_0\colon\beta_{\mathrm{newspaper}}=0$ własnymi słowami.
Jakie budżety są w tym porównaniu ustalone?

Odczytaj współczynnik prasy jako `press_coef` i jego błąd standardowy jako `press_se`.
Błąd standardowy opisuje niepewność wyznaczonego współczynnika.
Czy niezerowy wynik `press_coef` wystarcza do odrzucenia $H_0\colon\beta_{\mathrm{newspaper}}=0$?

<details>
<summary>Podpowiedź: błąd standardowy</summary>

Pole `bse` zawiera błędy standardowe. Wybierz z niego `"newspaper"` tak samo jak z `params`.

</details>

**Twoja odpowiedź:**

### 3.2. Ile błędów standardowych od zera?

Oblicz `t_press` jako iloraz współczynnika prasy i jego błędu standardowego.
Sprawdź wynik w `model_full.tvalues`. Odczytaj $p$-wartość dla prasy.

Przy poziomie $\alpha=0{,}05$ sformułuj wniosek o $H_0$.
Czy duża $p$-wartość dowodzi, że prasa nie wpływa na sprzedaż?

W teście $t$ dla $H_0\colon\beta_j=0$ liczymy
$$t=\frac{\widehat\beta_j}{\mathrm{SE}(\widehat\beta_j)}.$$
$p$-wartość opisuje, jak prawdopodobna byłaby przy prawdziwej $H_0$ statystyka
co najmniej tak odległa od zera jak otrzymana. Nie jest prawdopodobieństwem prawdziwości $H_0$.

<details>
<summary>Podpowiedź: wybrana p-wartość</summary>

`model_full.pvalues["newspaper"]` podaje p-wartość dwustronnego testu dla prasy.

</details>

**Twoja odpowiedź:**

### 3.3. Dwie cechy razem: porównaj sumy kwadratów reszt

Teraz pytamy, czy **radio i prasa wspólnie** wnoszą coś ponad TV.
Porównaj `model_tv` (model ograniczony) z `model_full` (model pełny).
Zapisz własnymi słowami hipotezę
$$H_0\colon\beta_{\mathrm{radio}}=\beta_{\mathrm{newspaper}}=0.$$

Odczytaj sumy kwadratów reszt `rss_tv`, `rss_full` i oblicz ich spadek.
Dlaczego samo zmniejszenie RSS nie wystarcza do odrzucenia $H_0$?
Czy test samej prasy odpowiada na to samo pytanie?

<details>
<summary>Podpowiedź: RSS</summary>

Pole `.ssr` dopasowanego modelu zawiera sumę kwadratów reszt (RSS).

</details>

**Twoja odpowiedź:**

### 3.4. Test F dla grupy współczynników

Porównujemy poprawę na jeden dodany współczynnik z oszacowanym poziomem szumu:
$$F=\frac{(\mathrm{RSS}_0-\mathrm{RSS})/q}{\mathrm{RSS}/(n-p-1)}.$$
Tutaj $\mathrm{RSS}_0$ to suma kwadratów reszt modelu z samym TV, a $\mathrm{RSS}$ -- modelu z TV, radiem i prasą.
Dodaliśmy dwie cechy ($q=2$), pełny model ma trzy cechy ($p=3$) i wyraz wolny,
a próba ucząca liczy $n=150$ obserwacji. Mianownik ma więc 146 stopni swobody.

Oblicz `F_group` ze wzoru. Sprawdź wynik przez porównanie modeli w bibliotece.
Odczytaj $p$-wartość i przy $\alpha=0{,}05$ sformułuj wniosek o **wspólnej** hipotezie.
Czy jej odrzucenie oznacza, że każdy z obu współczynników osobno jest różny od zera?

<details>
<summary>Podpowiedź 1: stopnie swobody</summary>

`model_full.df_resid` to liczba stopni swobody reszt w pełnym modelu.

</details>

<details>
<summary>Podpowiedź 2: porównanie z biblioteką</summary>

`model_full.compare_f_test(model_tv)` zwraca kolejno statystykę F, p-wartość i liczbę ograniczeń. Metodę wywołujemy na większym modelu, podając mniejszy jako argument.

</details>

**Twoja odpowiedź:**

W tym porównaniu TV pozostaje w obu modelach. Domyślne `model_full.fvalue` dotyczy
innego porównania: wszystkich trzech cech względem samej stałej.

**Opcjonalnie:** porównaj `model_full` z `model_add`, czyli usuń tylko prasę.
Sprawdź, czy statystyka F jest równa `t_press**2` i czy p-wartości są takie same.
To ta sama hipoteza o jednym współczynniku.

## 4. Interakcja: czy związek TV ze sprzedażą zależy od radia?

### 4.1. Rozszerz model o iloczyn

W części 2 przyrost TV o 10 tys. USD dawał taką samą różnicę predykcji przy obu budżetach radia.
Czy jest to odkrycie dotyczące danych, czy ograniczenie modelu addytywnego?

Dodaj do `data_tr` kolumnę `TV_radio` jako iloczyn budżetów TV i radia.
Dopasuj `model_int` zawierający `TV`, `radio` oraz `TV_radio`.
Zachowaj oba efekty główne. Odczytaj współczynniki.

Iloczyn jest nową cechą. Model nadal jest liniowy w swoich parametrach.

<details>
<summary>Podpowiedź: nowa kolumna</summary>

`data_tr["TV_radio"] = data_tr["TV"] * data_tr["radio"]` dodaje iloczyn w każdym wierszu.

</details>

**Twoja odpowiedź:**

### 4.2. Wróć do czterech rynków

Dodaj kolumnę iloczynu także do tabeli `markets`. Oblicz nowe predykcje jako `pred_int`
i ponownie porównaj przyrosty między wierszami 1 i 0 oraz 3 i 2.
Podaj różnice w sztukach. Przy którym budżecie radia różnica związana z TV jest większa?
Porównaj wynik z modelem addytywnym.

**Twoja odpowiedź:**

### Co zmieniło się w równaniu?

W modelu z interakcją możemy przegrupować składniki:
$$
\widehat y = \widehat\beta_0
+ (\widehat\beta_{\mathrm{TV}} + \widehat\beta_{\mathrm{TV\_radio}}\,\mathrm{radio})\,\mathrm{TV}
+ \widehat\beta_{\mathrm{radio}}\,\mathrm{radio}.
$$
Nachylenie względem TV wynosi teraz
$\widehat\beta_{\mathrm{TV}}+\widehat\beta_{\mathrm{TV\_radio}}\,\mathrm{radio}$.
Sam współczynnik przy TV opisuje nachylenie **przy zerowym budżecie radia**.
Nie jest jednym wspólnym nachyleniem dla wszystkich rynków.

### 4.3. Samodzielnie: trzeci budżet radia

Dla radia = 20 tys. USD oblicz ze współczynników różnicę przewidywanej sprzedaży
między TV = 100 i TV = 110 tys. USD. Sprawdź wynik przez `predict` i podaj go w sztukach.
Porównaj tę różnicę z różnicami predykcji tego samego modelu z interakcją dla budżetów radia 10 i 40 tys. USD, obliczonymi w 4.2. Czy leży między nimi? Wyjaśnij.

Następnie zapisz hipotezę o braku interakcji i odczytaj jej p-wartość.
Co odrzucenie tej hipotezy mówi o addytywności? Czy dowodzi skutku wspólnego zwiększenia wydatków?

Wykorzystaj wcześniejsze obliczenia i notatki; samodzielnie wybierz potrzebne operacje.

**Twoja odpowiedź:**

**Opcjonalnie:** narysuj dwie proste predykcji modelu z interakcją względem TV,
przy stałych budżetach radia 10 i 40. W każdym wierszu tabeli do predykcji musi być poprawny iloczyn budżetów.

## 5. Nowe dane i wnioski

### 5.1. Ocena predykcji

Dopiero teraz otwórz `data_te`. Dodaj do niego kolumnę iloczynu budżetów.
Oblicz testowe MSE czterech wcześniej ustalonych modeli:
średnia ucząca `mean_tr`, samo TV, TV + radio i model z interakcją.
Użyj dopasowanych już modeli; nie dopasowuj ich ponownie.

Przed uruchomieniem przewidź kolejność błędów. Czy lepsze dopasowanie uczące gwarantuje
lepszy wynik na tej próbie testowej?
Po obliczeniu wskaż, co w tej próbie pomaga przewidywać i jak duża jest poprawa.

Nie poszukujemy kolejnych modeli na podstawie wyników tego testu.
Wybranie najmniejszego z tych błędów i przedstawienie go jako niezależnej oceny procedury wyboru
powtarzałoby problem z labu 3.

<details>
<summary>Podpowiedź: nowe obserwacje</summary>

W `predict` podaj `data_te`, a błędy oblicz względem `data_te["sales"]`. Model stały nadal korzysta ze średniej z próby uczącej.

</details>

**Twoja odpowiedź:**

### Wnioski

Bez nowych obliczeń odpowiedz:

1. Co oznacza współczynnik przy TV w modelu addytywnym, a co w modelu z interakcją?
2. Czym różni się hipoteza testu $t$ dla prasy od hipotezy testu $F$ z części 3.4?
3. Czy mała $p$-wartość i mały błąd testowy mówią to samo? Do którego celu służy każda z tych informacji?

**Twoje odpowiedzi:**

## Fragmenty książki

Podstawą są sekcje rozdziału 6:

- [Model liniowy](https://rodakt.github.io/MAD_book/2-modele-liniowe/06-regresja-liniowa.html#sec-model-liniowy)
  i [metoda najmniejszych kwadratów](https://rodakt.github.io/MAD_book/2-modele-liniowe/06-regresja-liniowa.html#sec-mnk).
- [Wnioskowanie o parametrach](https://rodakt.github.io/MAD_book/2-modele-liniowe/06-regresja-liniowa.html#sec-wnioskowanie-mnk): błędy standardowe, test t i test F.
- [Interakcje i nieliniowość](https://rodakt.github.io/MAD_book/2-modele-liniowe/06-regresja-liniowa.html#sec-interakcje-nieliniowosc).

Jeśli chcesz wyprowadzić wzory na współczynniki prostej, przejdź do
[zadania 6.1 a)](https://rodakt.github.io/MAD_book/2-modele-liniowe/06-regresja-liniowa.html#exr-mnk-regresja-prosta).